# Deception Classifier — Colab GPU Training
Run each cell in order. Make sure the runtime is set to a GPU of your choice 
Runtime → Change runtime type → i.e., A100 GPU

## 1. Clone repo & install dependencies

In [12]:
!git clone https://github.com/Lucca878/deceptionClassifierTraining.git
%cd deceptionClassifierTraining

Cloning into 'deceptionClassifierTraining'...
remote: Enumerating objects: 191, done.
remote: Counting objects: 100% (191/191), done.
remote: Compressing objects: 100% (166/166), done.
remote: Total 191 (delta 23), reused 182 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (191/191), 4.32 MiB | 12.34 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/deceptionClassifierTraining/deceptionClassifierTraining


In [13]:
!pip install -q -r requirements.txt

## 2. Verify GPU

In [14]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

GPU available: True
Device: NVIDIA A100-SXM4-40GB


## 2.5 (Optional) Set hyperparameters to test
Set these BEFORE running CV. Leave as `None` to use defaults. Same parameters will be used for both CV and full training.

In [28]:
# Set hyperparameters to test (None = use defaults)
EPOCHS = 2          # int: number of epochs (default: 2)
LEARNING_RATE = 2e-5   # float: learning rate (default: 5e-5)
BATCH_SIZE = None      # int: batch size (default: 32)
WEIGHT_DECAY = None    # float: weight decay (default: 0.01)

# Example: to test with 3 epochs and higher LR:
# EPOCHS = 3
# LEARNING_RATE = 1e-4

print("Hyperparameters to test:")
print(f"  EPOCHS: {EPOCHS if EPOCHS is not None else '(default: 2)'}")
print(f"  LEARNING_RATE: {LEARNING_RATE if LEARNING_RATE is not None else '(default: 5e-5)'}")
print(f"  BATCH_SIZE: {BATCH_SIZE if BATCH_SIZE is not None else '(default: 32)'}")
print(f"  WEIGHT_DECAY: {WEIGHT_DECAY if WEIGHT_DECAY is not None else '(default: 0.01)'}")

Hyperparameters to test:
  EPOCHS: 2
  LEARNING_RATE: 2e-05
  BATCH_SIZE: (default: 32)
  WEIGHT_DECAY: (default: 0.01)


## 2.6 Select model

In [15]:
MODEL_KEY = "bert"  # Options: distilbert, bert, sbert, modernbert
print(f"Selected model: {MODEL_KEY}")

Selected model: bert


## 3. Run cross-validation first (tuning stage)

In [29]:
import os
import subprocess

# Build command with optional hyperparameters
cmd = [
    "python", "src/pipeline/run_pipeline.py",
    "--mode", "cv",
    "--model", MODEL_KEY,
    "--output_root", "./outputs",
    "--seed", "42"
]

if EPOCHS is not None:
    cmd.extend(["--epochs", str(EPOCHS)])
if LEARNING_RATE is not None:
    cmd.extend(["--lr", str(LEARNING_RATE)])
if BATCH_SIZE is not None:
    cmd.extend(["--batch_size", str(BATCH_SIZE)])
if WEIGHT_DECAY is not None:
    cmd.extend(["--weight_decay", str(WEIGHT_DECAY)])

print("Running CV with command:")
print(" ".join(cmd))
print()

# Stream child process output live in notebook
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)
for line in proc.stdout:
    print(line, end="")
return_code = proc.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, cmd)

Running CV with command:
python src/pipeline/run_pipeline.py --mode cv --model bert --output_root ./outputs --seed 42 --epochs 2 --lr 2e-05

Training model preset: bert
Backbone: bert-base-uncased
Loaded 4542 rows from data/hippocorpus/hippocorpus_training_truncated.csv

--- split_1 ---

Map: 100%|██████████| 3633/3633 [00:00<00:00, 6409.62 examples/s]

Map: 100%|██████████| 909/909 [00:00<00:00, 6427.83 examples/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3523.32it/s, Materializing param=bert.pooler.dense.weight]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     |

## 4. Inspect CV results (decide hyperparameters/model)

In [30]:
import os, glob

# Find latest run that actually contains CV results
run_dirs = sorted(glob.glob(f"outputs/{MODEL_KEY}_*"))
cv_run_dirs = [d for d in run_dirs if os.path.exists(os.path.join(d, "cv_results.csv"))]
latest = cv_run_dirs[-1] if cv_run_dirs else None

print("Latest CV run:", latest)

if latest:
    for f in os.listdir(latest):
        print(" ", f)
else:
    print("No CV run found yet. Run section 3 first.")

Latest CV run: outputs/bert_20260424_125559
  cv
  cv_results.csv
  config.json


In [ ]:
import pandas as pd
import os
import glob
import json

cv_path = os.path.join(latest, "cv_results.csv") if latest else None
if cv_path and os.path.exists(cv_path):
    # File is semicolon-separated
    df = pd.read_csv(cv_path, sep=";")
    print(df.head())
    print("\nRows, cols:", df.shape)

    # Main CV metric from stored predictions
    mean_acc = df["Correct"].mean()
    print("Mean CV accuracy:", round(mean_acc, 4))

    # Optional per-split and per-class breakdown
    print("\nAccuracy by split:")
    print(df.groupby("Split")["Correct"].mean().round(4))

    print("\nRecall by label (0=truthful, 1=deceptive):")
    print(df.groupby("Label")["Correct"].mean().round(4))

    # Compact validation + train summary from trainer logs
    eval_rows = []
    overfit_flags = []
    for split in [f"split_{i}" for i in range(1, 6)]:
        pattern = os.path.join(latest, "cv", split, "**", "trainer_state.json")
        state_files = sorted(glob.glob(pattern, recursive=True))
        if not state_files:
            continue

        with open(state_files[-1], "r") as f:
            state = json.load(f)

        log_history = state.get("log_history", [])
        eval_logs = [x for x in log_history if "eval_loss" in x]
        train_logs = [x for x in log_history if "loss" in x and "epoch" in x]
        if not eval_logs:
            continue

        # Collect eval points and nearest train loss up to that epoch
        for log in eval_logs:
            eval_epoch = float(log.get("epoch")) if log.get("epoch") is not None else None
            train_candidates = [
                t for t in train_logs
                if t.get("epoch") is not None and eval_epoch is not None and float(t.get("epoch")) <= eval_epoch
            ]
            train_loss = float(train_candidates[-1].get("loss")) if train_candidates else None

            eval_rows.append({
                "EvalLoss": float(log.get("eval_loss")),
                "EvalAccuracy": float(log.get("eval_accuracy")),
                "TrainLoss": train_loss,
            })

        # Overfitting flag per fold: last eval loss worse than best eval loss
        losses = [float(log.get("eval_loss")) for log in eval_logs]
        overfit_flags.append(losses[-1] > min(losses))

    if eval_rows:
        eval_df = pd.DataFrame(eval_rows)
        mean_train_loss = eval_df["TrainLoss"].dropna().mean()
        train_part = f", mean_train_loss={mean_train_loss:.4f}" if pd.notna(mean_train_loss) else ""
        print(
            "\nCompact validation summary: "
            f"mean_eval_loss={eval_df['EvalLoss'].mean():.4f}, "
            f"mean_eval_accuracy={eval_df['EvalAccuracy'].mean():.4f}"
            f"{train_part}, "
            f"overfit_folds={sum(overfit_flags)}/{len(overfit_flags)}"
        )

     Split  Prediction  Label  Correct
0  split_1           1      1     True
1  split_1           1      0    False
2  split_1           1      0    False
3  split_1           1      0    False
4  split_1           1      1     True

Rows, cols: (4542, 4)
Mean CV accuracy: 0.7517

Accuracy by split:
Split
split_1    0.6964
split_2    0.7536
split_3    0.7687
split_4    0.7588
split_5    0.7808
Name: Correct, dtype: float64

Recall by label (0=truthful, 1=deceptive):
Label
0    0.7509
1    0.7523
Name: Correct, dtype: float64

Compact validation summary: mean_eval_loss=0.5315, mean_eval_accuracy=0.7366, overfit_folds=0/5


In [ ]:
import glob
import json
import os
import pandas as pd

# Pull fold-level eval + train metrics (per epoch) from Trainer logs
epoch_rows = []

if latest:
    for split in [f"split_{i}" for i in range(1, 6)]:
        # Look for trainer_state.json in split root and nested checkpoints
        pattern = os.path.join(latest, "cv", split, "**", "trainer_state.json")
        state_files = sorted(glob.glob(pattern, recursive=True))
        if not state_files:
            continue

        state_path = state_files[-1]
        with open(state_path, "r") as f:
            state = json.load(f)

        log_history = state.get("log_history", [])
        eval_logs = [x for x in log_history if "eval_loss" in x]
        train_logs = [x for x in log_history if "loss" in x and "epoch" in x]

        for log in eval_logs:
            eval_epoch = float(log.get("epoch")) if log.get("epoch") is not None else None
            train_candidates = [
                t for t in train_logs
                if t.get("epoch") is not None and eval_epoch is not None and float(t.get("epoch")) <= eval_epoch
            ]
            train_loss = float(train_candidates[-1].get("loss")) if train_candidates else None

            epoch_rows.append({
                "Split": split,
                "Epoch": eval_epoch,
                "TrainLoss": train_loss,
                "EvalLoss": float(log.get("eval_loss")) if log.get("eval_loss") is not None else None,
                "EvalAccuracy": float(log.get("eval_accuracy")) if log.get("eval_accuracy") is not None else None,
            })

if epoch_rows:
    epoch_df = pd.DataFrame(epoch_rows).sort_values(["Split", "Epoch"])

    print("Train/validation metrics per epoch by split:")
    print(epoch_df.to_string(index=False))

    # Overfitting summary: last eval loss worse than best eval loss while train loss did not increase
    summary_rows = []
    for split, g in epoch_df.groupby("Split"):
        g = g.sort_values("Epoch")
        best_idx = g["EvalLoss"].idxmin()
        best = g.loc[best_idx]
        last = g.iloc[-1]

        train_improved_or_flat = (
            pd.notna(best["TrainLoss"]) and pd.notna(last["TrainLoss"]) and last["TrainLoss"] <= best["TrainLoss"]
        )
        eval_worsened = bool(last["EvalLoss"] > best["EvalLoss"])
        overfit_flag = bool(eval_worsened and train_improved_or_flat)

        summary_rows.append({
            "Split": split,
            "BestEpoch": best["Epoch"],
            "BestTrainLoss": best["TrainLoss"],
            "BestEvalLoss": best["EvalLoss"],
            "LastEpoch": last["Epoch"],
            "LastTrainLoss": last["TrainLoss"],
            "LastEvalLoss": last["EvalLoss"],
            "DeltaTrain(Last-Best)": (
                (last["TrainLoss"] - best["TrainLoss"])
                if pd.notna(best["TrainLoss"]) and pd.notna(last["TrainLoss"]) else None
            ),
            "DeltaEval(Last-Best)": last["EvalLoss"] - best["EvalLoss"],
            "PossibleOverfitting": overfit_flag,
        })

    summary_df = pd.DataFrame(summary_rows).sort_values("Split")
    print("\nOverfitting check by split:")
    print(summary_df.to_string(index=False))

    print("\nMean train_loss (at eval points):", round(epoch_df["TrainLoss"].dropna().mean(), 4))
    print("Mean eval_loss:", round(epoch_df["EvalLoss"].mean(), 4))
    print("Mean eval_accuracy:", round(epoch_df["EvalAccuracy"].mean(), 4))
    print("Folds flagged as possible overfitting:", int(summary_df["PossibleOverfitting"].sum()), "/", len(summary_df))
else:
    print("No eval logs found under fold directories.")
    print("Checked under:", os.path.join(latest, "cv"))
    print("Try re-running CV (section 3), then this cell.")

Validation metrics per epoch by split:
  Split  Epoch  EvalLoss  EvalAccuracy
split_1    1.0  0.637725      0.652365
split_1    2.0  0.579323      0.696370
split_2    1.0  0.548654      0.722772
split_2    2.0  0.503566      0.753575
split_3    1.0  0.502666      0.747797
split_3    2.0  0.480063      0.768722
split_4    1.0  0.540545      0.725771
split_4    2.0  0.508567      0.758811
split_5    1.0  0.521776      0.758811
split_5    2.0  0.492448      0.780837

Overfitting check by split:
  Split  BestEpoch  BestEvalLoss  LastEpoch  LastEvalLoss  DeltaLoss(Last-Best)  PossibleOverfitting
split_1        2.0      0.579323        2.0      0.579323                   0.0                False
split_2        2.0      0.503566        2.0      0.503566                   0.0                False
split_3        2.0      0.480063        2.0      0.480063                   0.0                False
split_4        2.0      0.508567        2.0      0.508567                   0.0                Fals

## 5. Train final model on full training set and zip it

In [ ]:
import os, glob, shutil, subprocess

# Fallback hyperparameters if section 2.5 wasn't run
try:
    _ = EPOCHS
except NameError:
    EPOCHS = None
try:
    _ = LEARNING_RATE
except NameError:
    LEARNING_RATE = None
try:
    _ = BATCH_SIZE
except NameError:
    BATCH_SIZE = None
try:
    _ = WEIGHT_DECAY
except NameError:
    WEIGHT_DECAY = None

# Build command with same hyperparameters used in CV
cmd = [
    "python", "src/pipeline/run_pipeline.py",
    "--mode", "full",
    "--model", MODEL_KEY,
    "--output_root", "./outputs",
    "--seed", "42"
]

if EPOCHS is not None:
    cmd.extend(["--epochs", str(EPOCHS)])
if LEARNING_RATE is not None:
    cmd.extend(["--lr", str(LEARNING_RATE)])
if BATCH_SIZE is not None:
    cmd.extend(["--batch_size", str(BATCH_SIZE)])
if WEIGHT_DECAY is not None:
    cmd.extend(["--weight_decay", str(WEIGHT_DECAY)])

print("Running full training with same hyperparameters as CV:")
print(" ".join(cmd))
print()

# Stream child process output live in notebook
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)
for line in proc.stdout:
    print(line, end="")
return_code = proc.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, cmd)

# Find newest full-training model folder
model_dirs = sorted(glob.glob(f"./outputs/{MODEL_KEY}_*/model", recursive=True))
if not model_dirs:
    raise FileNotFoundError(f"No trained model folder found under ./outputs/{MODEL_KEY}_*/model")

model_dir = model_dirs[-1]
zip_base = f"{MODEL_KEY}_retrained"
zip_file = zip_base + ".zip"

print("Using model dir:", model_dir)
shutil.make_archive(zip_base, "zip", model_dir)
print("Created zip:", zip_file)
print("Zip exists:", os.path.exists(zip_file), "| Size (MB):", round(os.path.getsize(zip_file) / 1024 / 1024, 2))

# Save the full path for section 6
ZIP_PATH = os.path.abspath(zip_file)
print(f"Zip file saved at: {ZIP_PATH}")

Running full training with same hyperparameters as CV:
python src/pipeline/run_pipeline.py --mode full --model distilbert --output_root ./outputs --seed 42

Using model dir: ./outputs/distilbert_20260424_113149/model
Created zip: distilbert_retrained.zip
Zip exists: True | Size (MB): 235.75


In [ ]:
from google.colab import drive
import shutil, os

# Use the path saved from section 5
src = ZIP_PATH if 'ZIP_PATH' in globals() else f"{MODEL_KEY}_retrained.zip"
assert os.path.exists(src), f"Missing file: {src}. Run section 5 first."

drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{MODEL_KEY}_trained.zip"
shutil.copy(src, dst)
print(f"Copied {src} to {dst}")
print(f"File size: {round(os.path.getsize(dst) / 1024 / 1024, 2)} MB")

Mounted at /content/drive
Copied distilbert_retrained.zip to /content/drive/MyDrive/distilbert_retrained.zip
File size: 235.75 MB
